In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def get_resnet18(num_classes=8):
  model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
  num_ftrs = model.fc.in_features
  model.fc = nn.Linear(num_ftrs, num_classes)
  return model

In [ ]:
from datacleaner_v2 import build_training_dataloaders

train_loader, val_loader = build_training_dataloaders(
    csv_path='./raw_data/train_labels_strId.csv', 
    img_dir='./raw_data/train/train', 
    batch_size=32
)

In [ ]:
import torch
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Instantiate both models
model_a = get_resnet18(num_classes=8).to(device)
model_b = get_resnet18(num_classes=8).to(device)

# 2. Load trained checkpoints (strip 'resnet.' prefix if needed)
state_a = torch.load(
    "./outputs/transfer_model_v4_phase2.pth", map_location=device
)
state_b = torch.load(
    "./outputs/resnet18_v6_b_phase2.pth", map_location=device
)

state_a = {k.replace("resnet.", "", 1): v for k, v in state_a.items()}
state_b = {k.replace("resnet.", "", 1): v for k, v in state_b.items()}

model_a.load_state_dict(state_a)
model_b.load_state_dict(state_b)

model_a.eval()
model_b.eval()

In [ ]:
# 3. Define our mapping dictionary (Numbers back to Strings)
idx_to_class = {
    0: 'CatEye', 1: 'CatFace', 2: 'CatMouth', 3: 'CatNose',
    4: 'DogEye', 5: 'DogFace', 6: 'DogMouth', 7: 'DogNose'
}

In [ ]:
from datacleaner_v2 import build_test_dataloader

test_loader, sub_df = build_test_dataloader(
    sample_sub_path='./sample_submission.csv', 
    img_dir='./raw_data/test/test', 
    batch_size=32
)

In [ ]:
import torch
import torch.nn.functional as F

print("Generating predictions on test data...")

# Set models to evaluation mode
model_a.eval()
model_b.eval()

predictions = []

with torch.no_grad():
  for images, ids in test_loader:
    images = images.to(device)

    # 1. Get class probabilities from both ResNet-18 models
    probs_a = F.softmax(model_a(images), dim=1)
    probs_b = F.softmax(model_b(images), dim=1)

    # 2. Average probabilities (50/50 soft voting)
    blended_probs = 0.5 * probs_a + 0.5 * probs_b

    # 3. Get predicted class names
    _, predicted_idx = torch.max(blended_probs, 1)
    for idx in predicted_idx.cpu().numpy():
      predictions.append(idx_to_class[idx])

# 4. Save to submission CSV
sub_df['class'] = predictions
sub_df.to_csv('./outputs/resnet_ensemble_submission.csv', index=False)
print(
    f"Done! Saved {len(predictions)} predictions to"
    " ./outputs/resnet_ensemble_submission.csv"
)